# Task 4: NLP Sentiment Classification Pipeline

This notebook implements the sentiment classification pipeline using the exact dataset schema (`review_id`, `review_text`, `sentiment`) and rows provided:
1. Loads `sentiment_reviews.csv` if present locally, downloads from the dataset URL if reachable, or writes your exact 10-row dataset (`review_id`, `review_text`, `sentiment`) to `sentiment_reviews.csv`.
2. Cleans and tokenizes `review_text` (HTML tag removal, punctuation stripping, lowercasing, stopword removal, and lemmatization).
3. Converts text into TF-IDF features using `TfidfVectorizer(max_features=5000)`.
4. Trains `MultinomialNB` and `LogisticRegression` on positive/negative reviews and prints accuracy benchmarks.
5. Provides `predict_sentiment()` for testing custom review phrases.

## 1. Setup & Imports

In [1]:
import io
import os
import re
import urllib.request
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

for resource in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"]:
    nltk.download(resource, quiet=True)

print("Libraries and NLTK resources loaded successfully.")

Libraries and NLTK resources loaded successfully.


## 2. Load / Download Dataset (`review_id`, `review_text`, `sentiment`)

This block checks for `sentiment_reviews.csv` locally, tries downloading from `https://www.datheon.in/datasets/sentiment_reviews.csv`, and if unavailable, saves your exact provided CSV data to `sentiment_reviews.csv` so the file exists on disk and loads cleanly.

In [2]:
RAW_CSV_DATA = '''review_id,review_text,sentiment
1,"The product quality is exceptional, exceeded all my expectations!",positive
2,"Terrible customer service and the item arrived completely damaged.",negative
3,"Average performance, decent for the price point but nothing special.",neutral
4,"Absolutely love this! Easy to setup, robust, and works flawlessly.",positive
5,"Stopped working after two days. Extremely disappointed with the purchase.",negative
6,"Customer support helped resolve my setup question within ten minutes.",positive
7,"Delivery was slow and package was missing one adapter cable.",negative
8,"Clean interface and intuitive controls. Highly recommended for students.",positive
9,"Not worth the money, plenty of better alternatives available.",negative
10,"Great value and durable build. Would definitely purchase again.",positive'''

DATASET_PATH = "sentiment_reviews.csv"
DATASET_URL = "https://www.datheon.in/datasets/sentiment_reviews.csv"

def load_sentiment_reviews(csv_path: str = DATASET_PATH, url: str = DATASET_URL) -> pd.DataFrame:
    if os.path.exists(csv_path):
        return pd.read_csv(csv_path)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=10) as resp:
            content = resp.read().decode("utf-8")
        # Verify it looks like CSV with review_text and sentiment rather than an HTML error page
        if "review_text" in content and "sentiment" in content:
            with open(csv_path, "w", encoding="utf-8", newline="") as f:
                f.write(content)
            print(f"Downloaded dataset from {url} to {csv_path}")
            return pd.read_csv(csv_path)
    except Exception as e:
        print(f"Remote download skipped ({e}). Saving exact provided dataset to {csv_path}...")

    with open(csv_path, "w", encoding="utf-8", newline="") as f:
        f.write(RAW_CSV_DATA.strip() + "\n")
    return pd.read_csv(csv_path)

df = load_sentiment_reviews()
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df

Downloaded dataset from https://www.datheon.in/datasets/sentiment_reviews.csv to sentiment_reviews.csv
Shape: (10, 3)
Columns: ['review_id', 'review_text', 'sentiment']


,review_id,review_text,sentiment
0,1,"The product quality is exceptional, exceeded a...",positive
1,2,Terrible customer service and the item arrived...,negative
2,3,"Average performance, decent for the price poin...",neutral
3,4,"Absolutely love this! Easy to setup, robust, a...",positive
4,5,Stopped working after two days. Extremely disa...,negative
5,6,Customer support helped resolve my setup quest...,positive
6,7,Delivery was slow and package was missing one ...,negative
7,8,Clean interface and intuitive controls. Highly...,positive
8,9,"Not worth the money, plenty of better alternat...",negative
9,10,Great value and durable build. Would definitel...,positive


### Filter to Positive (`1`) and Negative (`0`) Reviews
Your dataset includes a `neutral` row (`review_id` 3). Since the task asks to classify **positive and negative** sentiment (`1` = `positive`, `0` = `negative`), we filter to `positive` and `negative` rows for binary classification while keeping the full `df` intact above.

In [3]:
label_map = {"negative": 0, "positive": 1}
clf_df = df[df["sentiment"].str.strip().str.lower().isin(label_map.keys())].copy()
clf_df["label"] = clf_df["sentiment"].str.strip().str.lower().map(label_map)

print("Binary classification rows:", len(clf_df))
print(clf_df[["review_id", "review_text", "sentiment", "label"]])

Binary classification rows: 9
   review_id                                        review_text sentiment  \
0          1  The product quality is exceptional, exceeded a...  positive   
1          2  Terrible customer service and the item arrived...  negative   
3          4  Absolutely love this! Easy to setup, robust, a...  positive   
4          5  Stopped working after two days. Extremely disa...  negative   
5          6  Customer support helped resolve my setup quest...  positive   
6          7  Delivery was slow and package was missing one ...  negative   
7          8  Clean interface and intuitive controls. Highly...  positive   
8          9  Not worth the money, plenty of better alternat...  negative   
9         10  Great value and durable build. Would definitel...  positive   

   label  
0      1  
1      0  
3      1  
4      0  
5      1  
6      0  
7      1  
8      0  
9      1  


## 3. Text Cleaning & Tokenization

In [4]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english")) - {"not", "no", "nor", "neither", "never"}

def preprocess_text(text: str) -> str:
    text = re.sub(r"<[^>]+>", " ", str(text))
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = word_tokenize(text)
    cleaned_tokens = [
        lemmatizer.lemmatize(tok)
        for tok in tokens
        if tok not in stop_words and len(tok) > 1
    ]
    return " ".join(cleaned_tokens)

clf_df["cleaned_text"] = clf_df["review_text"].apply(preprocess_text)
clf_df[["review_id", "review_text", "cleaned_text", "label"]]

,review_id,review_text,cleaned_text,label
0,1,"The product quality is exceptional, exceeded a...",product quality exceptional exceeded expectation,1
1,2,Terrible customer service and the item arrived...,terrible customer service item arrived complet...,0
3,4,"Absolutely love this! Easy to setup, robust, a...",absolutely love easy setup robust work flawlessly,1
4,5,Stopped working after two days. Extremely disa...,stopped working two day extremely disappointed...,0
5,6,Customer support helped resolve my setup quest...,customer support helped resolve setup question...,1
6,7,Delivery was slow and package was missing one ...,delivery slow package missing one adapter cable,0
7,8,Clean interface and intuitive controls. Highly...,clean interface intuitive control highly recom...,1
8,9,"Not worth the money, plenty of better alternat...",not worth money plenty better alternative avai...,0
9,10,Great value and durable build. Would definitel...,great value durable build would definitely pur...,1


## 4. TF-IDF Feature Extraction & Model Training (`MultinomialNB` & `LogisticRegression`)

In [5]:
X = clf_df["cleaned_text"]
y = clf_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_preds)

lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_preds = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_preds)

benchmarks_df = pd.DataFrame({
    "Model": ["Multinomial Naive Bayes", "Logistic Regression"],
    "Test Accuracy": [round(nb_acc, 4), round(lr_acc, 4)],
    "Accuracy (%)": [round(nb_acc * 100, 2), round(lr_acc * 100, 2)]
})

display(benchmarks_df)
print("\nMultinomialNB Report:")
print(classification_report(y_test, nb_preds, target_names=["Negative", "Positive"], zero_division=0))
print("LogisticRegression Report:")
print(classification_report(y_test, lr_preds, target_names=["Negative", "Positive"], zero_division=0))

,Model,Test Accuracy,Accuracy (%)
0,Multinomial Naive Bayes,0.6667,66.67
1,Logistic Regression,0.6667,66.67



MultinomialNB Report:
              precision    recall  f1-score   support

    Negative       0.50      1.00      0.67         1
    Positive       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3

LogisticRegression Report:
              precision    recall  f1-score   support

    Negative       0.50      1.00      0.67         1
    Positive       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3



## 5. `predict_sentiment()` Helper Function

In [6]:
def predict_sentiment(text: str, model_type: str = "nb") -> dict:
    cleaned = preprocess_text(text)
    vec = vectorizer.transform([cleaned])
    model = nb_model if model_type.lower() in {"nb", "naive_bayes", "mnb"} else lr_model
    pred = int(model.predict(vec)[0])
    probs = model.predict_proba(vec)[0]
    return {
        "input_text": text,
        "cleaned_text": cleaned,
        "model": "MultinomialNB" if model is nb_model else "LogisticRegression",
        "sentiment": "Positive" if pred == 1 else "Negative",
        "confidence": round(float(probs[pred]), 4)
    }

for sample in [
    "The product is durable and easy to setup!",
    "Damaged item and slow delivery, not worth the money."
]:
    print(predict_sentiment(sample, "nb"))
    print(predict_sentiment(sample, "lr"))
    print("-" * 50)

{'input_text': 'The product is durable and easy to setup!', 'cleaned_text': 'product durable easy setup', 'model': 'MultinomialNB', 'sentiment': 'Positive', 'confidence': 0.6652}
{'input_text': 'The product is durable and easy to setup!', 'cleaned_text': 'product durable easy setup', 'model': 'LogisticRegression', 'sentiment': 'Positive', 'confidence': 0.5794}
--------------------------------------------------
{'input_text': 'Damaged item and slow delivery, not worth the money.', 'cleaned_text': 'damaged item slow delivery not worth money', 'model': 'MultinomialNB', 'sentiment': 'Negative', 'confidence': 0.6676}
{'input_text': 'Damaged item and slow delivery, not worth the money.', 'cleaned_text': 'damaged item slow delivery not worth money', 'model': 'LogisticRegression', 'sentiment': 'Negative', 'confidence': 0.584}
--------------------------------------------------
